# 📊 UAS Data Science — Week 3 Progress
## Modeling Awal: Baseline Models & Evaluasi

**Dataset:** Sales & Marketing Customer Dataset  
**Tujuan Week 3:** Membangun model baseline, menangani class imbalance dengan SMOTE, dan membandingkan performa tiga model klasifikasi

---

## 1. Recap Week 2

Pada Week 2 telah dilakukan:
- ✅ Cleaning: menghapus anomali, imputasi missing values → **14.997 baris bersih**
- ✅ Feature Engineering: 5 fitur baru (tenure, recency, spend_per_visit, engagement_score, refund_rate)
- ✅ Encoding: Label Encoding pada 7 kolom kategorik
- ✅ Scaling: StandardScaler pada 19 kolom numerik
- ✅ Output: `data_clean.csv` dan `data_modeling.csv`

**Week 3** fokus pada pembangunan model baseline dan evaluasinya.

---

## 2. Import Library & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, roc_curve
)
from imblearn.over_sampling import SMOTE

pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style='whitegrid', palette='Set2')

import warnings
warnings.filterwarnings('ignore')

print('✅ Library berhasil diimport')

In [ ]:
# Load data hasil preprocessing Week 2
df = pd.read_csv('data_modeling.csv')

X = df.drop(columns=['churn'])
y = df['churn']

print(f'Shape X: {X.shape}')
print(f'Shape y: {y.shape}')
print(f'Distribusi target:')
print(y.value_counts())
print(f'\nChurn rate: {y.mean()*100:.2f}%')

## 3. Train-Test Split

In [ ]:
# Stratified split 80:20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('=== Hasil Train-Test Split ===')
print(f'Training set : {X_train.shape[0]:,} baris ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Testing set  : {X_test.shape[0]:,} baris ({X_test.shape[0]/len(X)*100:.1f}%)')
print()
print('Distribusi churn pada training set:')
print(y_train.value_counts())
print()
print('Distribusi churn pada testing set:')
print(y_test.value_counts())

## 4. Penanganan Class Imbalance dengan SMOTE

In [ ]:
print('Distribusi SEBELUM SMOTE:')
print(pd.Series(y_train).value_counts())

# Terapkan SMOTE hanya pada training data
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print('\nDistribusi SETELAH SMOTE:')
print(pd.Series(y_train_sm).value_counts())
print(f'\nTotal training samples setelah SMOTE: {len(X_train_sm):,}')

In [ ]:
# Visualisasi distribusi sebelum vs sesudah SMOTE
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#2ecc71', '#e74c3c']
labels = ['Tidak Churn (0)', 'Churn (1)']

# Sebelum SMOTE
counts_before = pd.Series(y_train).value_counts().sort_index()
bars = axes[0].bar(labels, counts_before, color=colors, edgecolor='white', width=0.4)
for bar, val in zip(bars, counts_before):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{val:,}', ha='center', fontweight='bold')
axes[0].set_title('Sebelum SMOTE', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Jumlah Sampel')

# Setelah SMOTE
counts_after = pd.Series(y_train_sm).value_counts().sort_index()
bars = axes[1].bar(labels, counts_after, color=colors, edgecolor='white', width=0.4)
for bar, val in zip(bars, counts_after):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{val:,}', ha='center', fontweight='bold')
axes[1].set_title('Setelah SMOTE', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Jumlah Sampel')

plt.suptitle('Distribusi Target: Sebelum vs Sesudah SMOTE', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Training Baseline Models

### 5.1 Definisi Model

In [ ]:
# Definisi tiga model baseline
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree'      : DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
}

print('Model yang akan dilatih:')
for name in models:
    print(f'  - {name}')

### 5.2 Training & Evaluasi

In [ ]:
results = {}

for name, model in models.items():
    # Training dengan data SMOTE
    model.fit(X_train_sm, y_train_sm)

    # Prediksi pada test set (data asli tanpa SMOTE)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # Metrik evaluasi
    results[name] = {
        'Accuracy' : accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall'   : recall_score(y_test, y_pred),
        'F1-Score' : f1_score(y_test, y_pred),
        'ROC-AUC'  : roc_auc_score(y_test, y_prob),
        'y_pred'   : y_pred,
        'y_prob'   : y_prob
    }
    print(f'✅ {name} selesai dilatih')

print('\n✅ Semua model selesai dilatih')

## 6. Perbandingan Performa Model

In [ ]:
# Tabel perbandingan metrik
metrics_df = pd.DataFrame({
    name: {k: v for k, v in res.items() if k not in ['y_pred', 'y_prob']}
    for name, res in results.items()
}).T

print('=== Perbandingan Performa Model ===')
print(metrics_df.to_string())

# Highlight model terbaik
best_f1 = metrics_df['F1-Score'].idxmax()
best_auc = metrics_df['ROC-AUC'].idxmax()
print(f'\n🏆 Model terbaik berdasarkan F1-Score : {best_f1} ({metrics_df.loc[best_f1, "F1-Score"]:.4f})')
print(f'🏆 Model terbaik berdasarkan ROC-AUC  : {best_auc} ({metrics_df.loc[best_auc, "ROC-AUC"]:.4f})')

In [ ]:
# Bar chart perbandingan metrik
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
model_names  = list(results.keys())
x = np.arange(len(metric_names))
width = 0.25
colors_bar = ['#3498db', '#e67e22', '#2ecc71']

fig, ax = plt.subplots(figsize=(14, 6))

for i, (name, color) in enumerate(zip(model_names, colors_bar)):
    vals = [results[name][m] for m in metric_names]
    bars = ax.bar(x + i*width, vals, width, label=name, color=color, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(metric_names, fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Perbandingan Performa Semua Model', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

plt.tight_layout()
plt.show()

## 7. Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Tidak Churn', 'Churn'],
                yticklabels=['Tidak Churn', 'Churn'],
                linewidths=0.5, annot_kws={'size': 12})
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Prediksi')
    ax.set_ylabel('Aktual')

plt.suptitle('Confusion Matrix — Semua Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. ROC Curve

In [ ]:
plt.figure(figsize=(9, 7))
colors_roc = ['#3498db', '#e67e22', '#2ecc71']

for (name, res), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    auc_val = res['ROC-AUC']
    plt.plot(fpr, tpr, color=color, linewidth=2.5,
             label=f'{name} (AUC = {auc_val:.4f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Random Classifier')
plt.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve — Perbandingan Semua Model', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.tight_layout()
plt.show()

## 9. Classification Report Detail

In [ ]:
for name, res in results.items():
    print(f'{'='*50}')
    print(f' {name}')
    print(f'{'='*50}')
    print(classification_report(y_test, res['y_pred'],
                                 target_names=['Tidak Churn', 'Churn']))
    print()

## 10. Feature Importance (Random Forest)

In [ ]:
# Feature importance dari Random Forest
rf_model = models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
top20 = importances.sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 8))
colors_fi = sns.color_palette('RdYlGn_r', 20)
bars = plt.barh(top20.index[::-1], top20.values[::-1], color=colors_fi[::-1])
for bar, val in zip(bars, top20.values[::-1]):
    plt.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=9)
plt.xlabel('Feature Importance Score')
plt.title('Top 20 Feature Importance — Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nTop 10 fitur terpenting:')
print(top20.head(10).to_string())

## 11. Cross-Validation (5-Fold)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, model in models.items():
    cv_scores = cross_val_score(model, X_train_sm, y_train_sm,
                                cv=skf, scoring='f1', n_jobs=-1)
    cv_results[name] = cv_scores
    print(f'{name}:')
    print(f'  F1 per fold : {[f"{s:.4f}" for s in cv_scores]}')
    print(f'  Mean ± Std  : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
    print()

In [ ]:
# Visualisasi cross-validation
fig, ax = plt.subplots(figsize=(10, 5))

cv_df = pd.DataFrame(cv_results)
cv_df.index = [f'Fold {i+1}' for i in range(5)]

for col, color in zip(cv_df.columns, ['#3498db', '#e67e22', '#2ecc71']):
    ax.plot(cv_df.index, cv_df[col], marker='o', linewidth=2,
            markersize=8, color=color, label=col)

ax.set_title('Cross-Validation F1-Score per Fold', fontsize=13, fontweight='bold')
ax.set_ylabel('F1-Score')
ax.set_ylim(0, 1)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## 12. Ringkasan Week 3

### ✅ Yang Telah Dilakukan

| Tahap | Detail |
|-------|--------|
| **Train-Test Split** | Stratified 80:20 → training 11.997, testing 3.000 baris |
| **SMOTE** | Menyeimbangkan training data dari ~84:16 menjadi 50:50 |
| **Logistic Regression** | Model baseline linear |
| **Decision Tree** | Model berbasis rule (max_depth=5) |
| **Random Forest** | Ensemble method (100 estimators) |
| **Cross-Validation** | 5-Fold StratifiedKFold untuk validasi kestabilan model |

### 📊 Kesimpulan Sementara

- **Random Forest** cenderung memberikan performa terbaik secara keseluruhan berdasarkan F1-Score dan ROC-AUC
- `recency_days`, `lifetime_value`, dan `total_spent` termasuk fitur paling penting
- Recall untuk kelas Churn perlu diperhatikan karena false negative (pelanggan churn tidak terdeteksi) lebih merugikan bisnis

### 🗓️ Rencana Week 4
- Hyperparameter tuning dengan **GridSearchCV / RandomizedSearchCV**
- Uji model tambahan: **XGBoost / Gradient Boosting**
- Optimasi threshold klasifikasi untuk memaksimalkan Recall pada kelas Churn
- Analisis error: false positives & false negatives